In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# 1. 应用你指定的绘图参数
plt.rcParams.update({
    "axes.linewidth": 1.5,
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5,
    "xtick.major.size": 8,
    "ytick.major.size": 8,
    "font.size": 14,
    "axes.labelsize": 14,
    "figure.figsize": (8, 6)
})

# 2. 文件夹配置
base_dir = "."  # 假设你在 12nt_sim 目录下运行 notebook
num_seeds = 10
filename = "lettuce_hbonds.dat"

all_data = []

# 3. 读取所有文件夹中的数据
print("Loading hydrogen bond data...")
for i in range(1, num_seeds + 1):
    folder_name = f"run_seed_{i}"
    file_path = os.path.join(base_dir, folder_name, filename)
    
    if os.path.exists(file_path):
        # oxDNA 文件通常以 # 开头作为注释，使用 sep='\s+' 处理空格分隔
        data = pd.read_csv(file_path, sep='\s+', comment='#', names=['Step', 'HB_Count'])
        all_data.append(data.set_index('Step'))
    else:
        print(f"Warning: {file_path} not found.")

# 4. 合并数据并计算均值与标准差 (Error Bar)
# 将所有 dataframe 合并，按 Step 对齐
df_combined = pd.concat(all_data, axis=1)

# 计算横向（跨 seed）的均值和标准差
mean_hb = df_combined.mean(axis=1)
std_hb = df_combined.std(axis=1)  # 如果你想要标准误，可以用 .sem(axis=1)
steps = mean_hb.index

# 5. 绘图
plt.figure()

# 绘制带有 Error Bar 的图
# 为了防止点太密集导致 Error bar 变成色块，我们可以使用 errorevery 参数
plt.errorbar(steps, mean_hb, yerr=std_hb, 
             fmt='-o', color='#2c3e50', ecolor='#e74c3c', 
             elinewidth=1, capsize=2, ms=4, label='Mean HB Count',
             errorevery=1000000) # 每 10 个点画一个 error bar，根据你的 print_every 调整

plt.xlabel("Simulation Steps")
plt.ylabel("Number of Hydrogen Bonds")
plt.ylim(0, 13)  # 12 nt stem，上限设为 13 比较美观
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

# 打印最终统计结果
print(f"Final Average HB Count: {mean_hb.iloc[-1]:.2f} ± {std_hb.iloc[-1]:.2f}")

In [ ]:
# --- Section 2: Aptamer Stem Pair Distance Analysis ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# 继续使用你定义的全局绘图参数
plt.rcParams.update({
    "axes.linewidth": 1.5,
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5,
    "xtick.major.size": 8,
    "ytick.major.size": 8,
    "font.size": 14,
    "axes.labelsize": 14,
})

# 1. 参数设置
base_dir = "."
num_seeds = 10
filename = "distance.dat"
nm_conversion = 0.8518  # oxDNA2 长度单位转 nm
num_pairs = 12

all_seed_means = []
all_seed_vars = []

print("Processing distance files from all seeds...")

# 2. 遍历读取并提取统计量
for i in range(1, num_seeds + 1):
    file_path = os.path.join(base_dir, f"run_seed_{i}", filename)
    if os.path.exists(file_path):
        # 读取数据：无表头，第0列是Step，第1-12列是Pair距离
        data = pd.read_csv(file_path, sep='\s+', header=None, comment='#')
        
        # 提取 1-12 列数据并转换为 nm
        dist_data_nm = data.iloc[:, 1:13] * nm_conversion
        
        # 计算该 seed 的均值和方差（存入列表以便后续汇总）
        all_seed_means.append(dist_data_nm.mean())
        all_seed_vars.append(dist_data_nm.var())
    else:
        print(f"Warning: {file_path} not found.")

# 3. 汇总 10 个副本的统计数据 (Ensemble Average)
# 转换为 DataFrame: 行是 seed (0-9), 列是 Pair (1-12)
means_df = pd.DataFrame(all_seed_means)
vars_df = pd.DataFrame(all_seed_vars)

# 计算总均值 (Mean of means)
final_means = means_df.mean()

# 计算总标准差 (结合了时间波动和种子间差异)
# 公式：总方差 = 组内方差的平均 + 组间均值的方差
final_vars = vars_df.mean() + means_df.var()
final_stds = np.sqrt(final_vars)

# 4. 绘图
pair_indices = np.arange(1, num_pairs + 1)

plt.figure(figsize=(9, 6))

plt.errorbar(pair_indices, final_means, yerr=final_stds, 
             fmt='-o', color='#34495e', ecolor='#e67e22', 
             elinewidth=2, capsize=6, ms=8, mfc='white', mew=2,
             label='Ensemble Mean Distance')

# 图表修饰
plt.xlabel("Base Pair Index (36-47 / 94-83)")
plt.ylabel("Inter-strand Distance (nm)")
plt.xticks(pair_indices) # 确保 1-12 每个刻度都显示
plt.grid(True, axis='y', linestyle=':', alpha=0.6)
plt.title("Aptamer Stem Stability Analysis", pad=15)
plt.legend(frameon=False)

# 设置 y 轴范围，使趋势更清晰（根据你的数据微调，通常 DNA 间距在 1nm 左右）
plt.ylim(0.8, 1.4) 

plt.tight_layout()
plt.show()

# 5. 打印汇总表（可选，方便你写论文查数）
summary_df = pd.DataFrame({
    'Pair_Num': pair_indices,
    'Mean_nm': final_means.values,
    'Std_nm': final_stds.values
})
print("\n--- Summary Statistics ---")
print(summary_df.to_string(index=False))

In [ ]:
# --- Section 3: Stem Total Hydrogen Bonding Energy Analysis ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# 1. 应用指定的绘图参数
plt.rcParams.update({
    "axes.linewidth": 1.5,
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5,
    "xtick.major.size": 8,
    "ytick.major.size": 8,
    "font.size": 14,
    "axes.labelsize": 14,
    "figure.figsize": (10, 6)
})

base_dir = "."
num_seeds = 10
filename = "stem_total_hb_energy.dat"
bin_size = 10000000  # 每 1e6 步打一个点

all_seeds_binned = []

print("Processing HB energy files and binning data...")

# 2. 读取并预处理数据
for i in range(1, num_seeds + 1):
    file_path = os.path.join(base_dir, f"run_seed_{i}", filename)
    if os.path.exists(file_path):
        # 读取数据：Step, HB_Energy
        data = pd.read_csv(file_path, sep='\s+', header=None, comment='#', names=['Step', 'Energy'])
        
        # 核心逻辑：按 1e6 的区间进行分组并取均值
        # 这样可以将 10万个点压缩成 100个点
        data['Bin'] = (data['Step'] // bin_size) * bin_size
        binned = data.groupby('Bin')['Energy'].mean()
        all_seeds_binned.append(binned)
    else:
        print(f"Warning: {file_path} not found.")

# 3. 计算 10 个 seed 之间的统计量
df_energy_combined = pd.concat(all_seeds_binned, axis=1)
mean_energy = df_energy_combined.mean(axis=1)
std_energy = df_energy_combined.std(axis=1)
binned_steps = mean_energy.index

# 4. 绘图
plt.figure()

# 绘制能量随时间变化的折线图
# 能量通常是负值，负得越多越稳定
plt.errorbar(binned_steps, mean_energy, yerr=std_energy, 
             fmt='-s',               # 使用方块 marker 区分
             color='#8e44ad',        # 紫色线条
             ecolor='#9b59b6',       # 淡紫色误差棒
             elinewidth=1.5, 
             capsize=3, 
             ms=5, 
             alpha=0.8,
             label='Mean Total HB Energy ± SD')

# 图表修饰
plt.xlabel("Simulation Steps")
plt.ylabel("Total HB Energy (oxDNA units)")
plt.grid(True, linestyle=':', alpha=0.5)

# 科学计数法处理横轴
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0))

# 建议：根据你的 14.89 观测值，设置合适的 y 轴范围
# 这里的 12 对碱基能量通常在 -14 到 -18 左右
current_min = mean_energy.min()
plt.ylim(current_min - 2, 0) # 从稍微低于最小值到 0

plt.legend(frameon=False, loc='best')
plt.tight_layout()
plt.show()

# 5. 打印统计概要
print(f"Global Average HB Energy: {mean_energy.mean():.3f} ± {std_energy.mean():.3f} oxDNA units")